# 04 — How the Master Training CSV Was Built

This notebook documents the data pipeline that produces `data/master_dataset.csv`.
You don't need to run this notebook to use the model — the CSV is committed in the
repo. But you should read it if you want to:

- Extend the dataset with new structures
- Recompute features after a code change in `src/geometry.py`
- Audit the data provenance for the report

## The three data sources

| Source | n | What it has |
|---|---|---|
| **704 systematic** | 704 | Idealised Fe-porphine dimer scans along $d_z$, $d_\mathrm{lat}$, $\theta_\mathrm{tilt}$, $\varphi_\mathrm{twist}$, all at the SRSH-PCM/CDFT-CI level |
| **111 Slater MD** | 111 | MD-sampled Fe-porphine dimers from the `md160_goodconv` set (Slater group); same level of theory |
| **15 MD validation** | 15 | Fe-TPP + peptoid MD snapshots with $\Gamma^*$ values from Bakenhaster et al. |

For each, we have the dimer xyz, the reference $J$ (CDFT-CI in eV), and various
electronic features computed in earlier notebooks.

## The columns of the master CSV

| Column | Source | Notes |
|---|---|---|
| `name`, `dataset`, `split` | metadata | identifier and 80/20 train/test or 'validation' |
| `J_cdft_eV`, `log10_J_cdft` | reference | the label we predict |
| `dz, dx, dy, cos_theta, ...` (11 cols) | `geometry.dimer_features()` | symmetry-adapted features |
| `S12_b3lyp` | B3LYP/6-31G* HOMO via molden | original v4 ansatz |
| `S12_srsh_singlet` | SRSH cation HOMO via fchk | rotation-aware, neutral monomer |
| `S12_srsh_cation` | SRSH cation $\alpha$-SOMO via fchk | rotation-aware, **production feature** |
| `S12_qchem_truth` | Q-Chem CDFT-CI dimer | ground-truth (only for 111 Slater) |
| `J_eht_eV`, `J_zindo_eV`, `J_dftb_eV` | dimer-level cheap methods | tested-and-rejected (notebook 07) |


## Setup

In [1]:
import sys, os, json
import numpy as np
import pandas as pd

sys.path.insert(0, '../src')


## The build script

The full pipeline that produces the CSV from the various raw `.npy` and `.npz`
caches is in `src/build_master_csv.py`. Here we show its logic step by step.

In [2]:
DATA = '../data/processed'

# Inspect what raw files are available
print("Raw caches in data/processed/ used to build the master CSV:")
for f in sorted(os.listdir(DATA)):
    full = os.path.join(DATA, f)
    if not os.path.isfile(full): continue
    size = os.path.getsize(full)
    print(f"  {f:<35} {size:>10} bytes")


Raw caches in data/processed/ used to build the master CSV:
  dftb_J_30_smoke.npz                       3198 bytes
  eht_J_704.npz                            75815 bytes
  eht_J_md.npz                              2160 bytes
  eht_J_slater.npz                          7476 bytes
  features.npy                             45184 bytes
  features_md.npy                           1448 bytes
  features_slater.npy                       9896 bytes
  features_v2.npy                          62080 bytes
  features_v3.npy                          73344 bytes
  features_v3a.npy                         95872 bytes
  features_v3b.npy                         95872 bytes
  features_v4_dft.npy                      78976 bytes
  features_v4_molden.npy                   78976 bytes
  homo_coeffs_molden.npy                    3280 bytes
  homo_coeffs_srsh_cation.npy               3512 bytes
  homo_coeffs_srsh_cation_sph.npy           3280 bytes
  homo_coeffs_srsh_singlet.npy              3512 bytes
  hom

### 704 systematic block

In [3]:
meta = pd.read_csv(f'{DATA}/metadata.csv')
features_v2 = np.load(f'{DATA}/features_v2.npy')   # (704, 11)
labels_ev   = np.load(f'{DATA}/labels_ev.npy')     # (704,)

print(f'Metadata: {len(meta)} rows, columns = {list(meta.columns)}')
print(f'Features shape: {features_v2.shape}')
print(f'Coupling range (eV): [{labels_ev.min():.2e}, {labels_ev.max():.2f}]')
print(f'Train/test split: {meta["split"].value_counts().to_dict()}')


Metadata: 704 rows, columns = ['idx', 'filename', 'coupling_eV', 'log10_coupling', 'is_near_zero', 'split']
Features shape: (704, 11)
Coupling range (eV): [0.00e+00, 0.39]
Train/test split: {'train': 563, 'test': 141}


### 111 Slater MD block

In [4]:
slater_recs = json.load(open(f'{DATA}/slater_records.json'))
features_slater = np.load(f'{DATA}/features_slater.npy')
slater_S12_qchem = np.load(f'{DATA}/slater_S12_qchem.npy')

print(f'Slater records: {len(slater_recs)}')
print(f'Sample record: {slater_recs[0]}')
print(f'Features shape: {features_slater.shape}')
print(f'CDFT-CI |S_12| ground-truth available for {(slater_S12_qchem != 0).sum()} of {len(slater_S12_qchem)}')


Slater records: 111
Sample record: {'name': 'md_003', 'J_eV': 0.0035538070437260328, 'h12_ha': -0.0001306}
Features shape: (111, 11)
CDFT-CI |S_12| ground-truth available for 111 of 111


### 15 MD validation block

These have $\Gamma^*$ reference values from Bakenhaster et al. Two of them
(conformations 9 and 15) have $\Gamma^* = 0$ exactly because of $D_{4h}$
symmetry nodes (the lateral displacement happens to coincide with an interference
zero of the porphyrin $\pi$-system overlap).

In [5]:
GAMMA_STAR_MEV = {
    1: 39.9246, 2: 0.1905, 3: 5.4858, 4: 5.5266, 5: 42.1205,
    6: 7.6328, 7: 22.4902, 8: 30.6074, 9: 0.0000, 10: 0.0136,
    11: 0.5987, 12: 0.3755, 13: 0.0190, 14: 10.1553, 15: 0.0000,
}
features_md = np.load(f'{DATA}/features_md.npy')
print(f'MD features shape: {features_md.shape}')
print(f'\nGamma* values (mV) for the 15 conformations:')
for k, v in GAMMA_STAR_MEV.items():
    print(f'  conf_{k:02d}: {v:7.3f}{"  <-- geometric node" if v == 0 else ""}')


MD features shape: (15, 11)

Gamma* values (mV) for the 15 conformations:
  conf_01:  39.925
  conf_02:   0.191
  conf_03:   5.486
  conf_04:   5.527
  conf_05:  42.120
  conf_06:   7.633
  conf_07:  22.490
  conf_08:  30.607
  conf_09:   0.000  <-- geometric node
  conf_10:   0.014
  conf_11:   0.599
  conf_12:   0.376
  conf_13:   0.019
  conf_14:  10.155
  conf_15:   0.000  <-- geometric node


## Final assembly and verification

In [6]:
# Load the assembled master CSV and sanity check
df = pd.read_csv('../data/master_dataset.csv')
print(f'Master CSV: {len(df)} rows × {len(df.columns)} columns')
print()
print('Per-dataset row counts:')
print(df['dataset'].value_counts())
print()
print('Missing values per column (NaN counts):')
nans = df.isna().sum()
print(nans[nans > 0])


Master CSV: 830 rows × 23 columns

Per-dataset row counts:
dataset
704_systematic    704
slater_111_md     111
md_validation      15
Name: count, dtype: int64

Missing values per column (NaN counts):
log10_J_cdft        69
S12_b3lyp           15
S12_qchem_truth    719
J_zindo_eV          74
J_dftb_eV          804
dtype: int64


### Why some columns have NaN

- `S12_b3lyp` is NaN for the 15 MD validation set: we never bothered computing
  the original B3LYP-molden ansatz on those (we have the better SRSH ansatz).
- `S12_qchem_truth` is NaN for 704 systematic and MD validation: the Q-Chem
  CDFT-CI raw $|S_{12}|$ is only available for the 111 Slater set.
- `J_dftb_eV` is mostly NaN: the DFTB+ run was a 30-structure smoke test only
  (see notebook 07 for why).

These NaNs are intentional — they mark "feature not computed for this dataset",
not data corruption.

## Reproducibility

To regenerate the CSV from raw outputs:

```bash
python src/build_master_csv.py
```

This re-reads all the `.npy` / `.npz` / `.csv` files in `data/processed/` and
writes `data/master_dataset.csv`. It is deterministic given the same inputs.
